# RIPPER DEMO za dataset *Otkrivanje transakcijskih prijevara*

https://www.kaggle.com/competitions/dap-fer2025

# 0. Priprema dataseta (ponavljanje od ranije)

In [2]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#df = pd.read_csv('dataset.csv')
df = pd.read_csv(
    "https://raw.githubusercontent.com/nfridFER/DAP/main/DEMO/dataset.csv"
)

# brisanje značajke Is_Contactless
del df['Is_Contactless']

# uklanjanje zapisa gdje nedostaju podaci, svi od USER_0007
df = df[df['User_ID'] != 'USER_0007'].copy()

# korekcija neujednačenog označavanja
df['Location'] = df['Location'].replace('Lndn', 'London')
df['Merchant_Category'] = df['Merchant_Category'].replace('Electronis', 'Electronics')

# izbacivanje negativnih i transakcija s iznosom 0
df = df.loc[df['Transaction_Amount'] > 0, :].copy()

# popravljanje Is_Weekend
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df['Is_Weekend'] = df["Timestamp"].dt.dayofweek >= 5

# one-hot encoding
df = pd.get_dummies(df, columns=['Transaction_Type'], prefix='TT')
df = pd.get_dummies(df, columns=['Device_Type'], prefix='DT')
df = pd.get_dummies(df, columns=['Location'], prefix='Loc')
df = pd.get_dummies(df, columns=['Merchant_Category'], prefix='MerchCat')
df = pd.get_dummies(df, columns=['Card_Type'], prefix='CT')
df = pd.get_dummies(df, columns=['Authentication_Method'], prefix='AUM')

# log transform
df['Transaction_Amount'] = np.log1p(df['Transaction_Amount'])

# date transform
df["Day"] = df["Timestamp"].dt.day
df["Month"] = df["Timestamp"].dt.month
df["Year"] = df["Timestamp"].dt.year
df["Day_of_Week"] = df["Timestamp"].dt.dayofweek
df["Quarter"] = df["Timestamp"].dt.quarter

df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
df["DOW_sin"] = np.sin(2 * np.pi * df["Day_of_Week"] / 7)
df["DOW_cos"] = np.cos(2 * np.pi * df["Day_of_Week"] / 7)
df["Day_sin"] = np.sin(2 * np.pi * df["Day"] / 31)
df["Day_cos"] = np.cos(2 * np.pi * df["Day"] / 31)

del df['Timestamp']
del df['Day_of_Week']
del df['Day']
del df['Month']
del df['Year']
del df['Quarter']

# ekspertne značajke
df["Amount_vs_Avg7d"] = df["Transaction_Amount"] / (df["Avg_Transaction_Amount_7d"] + 1)
df["Txn_Burst"] = (df["Daily_Transaction_Count"] > 10).astype(int)
df["Weekend_Online"] = df["Is_Weekend"] * df["TT_Online"]



y = df["Fraud_Label"].copy()
X = df.drop(columns=['Transaction_ID', 'User_ID', 'Fraud_Label']).copy()

# bool -> int
bool_cols = X.select_dtypes(include=['bool']).columns
X.loc[:, bool_cols] = X.loc[:, bool_cols].astype(int)





/tmp/ipykernel_1619/2014952700.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  X.loc[:, bool_cols] = X.loc[:, bool_cols].astype(int)
/tmp/ipykernel_1619/2014952700.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  X.loc[:, bool_cols] = X.loc[:, bool_cols].astype(int)
/tmp/ipykernel_1619/2014952700.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 1 0 ... 0 1 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  X.loc[:, bool_cols] = X.loc[:, bool_cols].astype(int)
/tmp/ipykernel_1619/2014952700.py:71: FutureWarni

# 1. 5-fold CV - performanse

In [11]:
#!pip install wittgenstein
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import wittgenstein as lw


features_A = [
    'Failed_Transaction_Count_7d',
    'Risk_Score',
    'Daily_Transaction_Count',
    'Loc_New York',
    'AUM_PIN',
    'Day_sin',
    'MerchCat_Restaurants',
    'DT_Tablet',
    'Avg_Transaction_Amount_7d',
    'CT_Discover'
]

features_B = [
    'Failed_Transaction_Count_7d',
    'Risk_Score'
]



from sklearn.model_selection import StratifiedKFold

def cross_validate_ripper(X, y, features, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    accs, precs, recs, f1s = [], [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model = lw.RIPPER()
        model.fit(X_tr[features], y_tr)

        y_pred = model.predict(X_te[features])

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred))
        recs.append(recall_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred))

    print(f"\n=== 5-FOLD CV ({features}) ===")
    print("Accuracy :", np.mean(accs))
    print("Precision:", np.mean(precs))
    print("Recall   :", np.mean(recs))
    print("F1       :", np.mean(f1s))


cross_validate_ripper(X, y, features_A)
cross_validate_ripper(X, y, features_B)


=== 5-FOLD CV (['Failed_Transaction_Count_7d', 'Risk_Score', 'Daily_Transaction_Count', 'Loc_New York', 'AUM_PIN', 'Day_sin', 'MerchCat_Restaurants', 'DT_Tablet', 'Avg_Transaction_Amount_7d', 'CT_Discover']) ===
Accuracy : 0.9606384498449845
Precision: 0.8928290956665792
Recall   : 0.9973241872286494
F1       : 0.9421578102588274

=== 5-FOLD CV (['Failed_Transaction_Count_7d', 'Risk_Score']) ===
Accuracy : 0.9606184478447846
Precision: 0.8910214831373173
Recall   : 0.9997510696181952
F1       : 0.9422582838312611


# 2. Obični split za analizu pravila

In [6]:
from sklearn.model_selection import train_test_split

def evaluate_model(model, X_train, y_train, X_test, y_test, name="MODEL"):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # wittgenstein gives probs via predict_proba
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    except:
        auc = None

    print(f"\n=== {name} ===")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1       :", f1_score(y_test, y_pred))
    if auc:
        print("ROC AUC  :", auc)

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nRules:")
    print(model.ruleset_)


X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )


# A
model_A = lw.RIPPER()
evaluate_model(
    model_A,
    X_train[features_A], y_train,
    X_test[features_A], y_test,
    name="RIPPER A"
)

# B
model_B = lw.RIPPER()
evaluate_model(
    model_B,
    X_train[features_B], y_train,
    X_test[features_B], y_test,
    name="RIPPER B"
)




=== RIPPER A ===
Accuracy : 0.9623
Precision: 0.8952354416271942
Recall   : 0.9996888612321095
F1       : 0.9445832720858445
ROC AUC  : 0.9960208533583639

Confusion matrix:
[[6410  376]
 [   1 3213]]

Rules:
[[Failed_Transaction_Count_7d=4.0] V [Risk_Score=>0.9] V [Risk_Score=0.8-0.9^AUM_PIN=0] V [Risk_Score=0.8-0.9]]

=== RIPPER B ===
Accuracy : 0.9623
Precision: 0.8952354416271942
Recall   : 0.9996888612321095
F1       : 0.9445832720858445
ROC AUC  : 0.995867072128257

Confusion matrix:
[[6410  376]
 [   1 3213]]

Rules:
[[Failed_Transaction_Count_7d=4.0] V [Risk_Score=>0.9] V [Risk_Score=0.8-0.9^Failed_Transaction_Count_7d=1.0] V [Risk_Score=0.8-0.9^Failed_Transaction_Count_7d=0.0] V [Risk_Score=0.8-0.9]]


*Uočiti redundaciju: ```[Risk_Score=0.8-0.9^AUM_PIN=0] V [Risk_Score=0.8-0.9]]``` -->(A AND B) OR A = A*

*RIPPER heuristike ne garantiraju pronalazak minimalnog seta*